In [1]:

%run encode.ipynb

         asin                                              title  price_eur  \
0  B0G71BJS8S  Microsoft Surface Laptop, 13.8" | Snapdragon X...    1056.07   
1  B0DYDVFVTJ  Microsoft Surface Laptop, 13" | Snapdragon X P...     735.54   
2  B0F8L98RLY  HP Laptop | 15,6" FHD Display | Intel N100 | 4...     219.00   
3  B0H28V9JGR  Laptop 15,6 Zoll 8GB RAM 256GB SSD Notebook Fu...     249.19   
4  B0DJBQ8Y7K  HP Chromebook x360 Laptop (14" HD Touchscreen,...     219.00   

   rating  reviews  ram_gb  storage_gb  screen_size_inch  \
0     4.8     15.0    16.0       512.0              13.8   
1     4.3     74.0    16.0       256.0              13.0   
2     4.1    170.0     4.0       128.0              15.6   
3     4.2      8.0     8.0       256.0              15.6   
4     4.2    181.0     4.0       128.0              14.0   

                                         product_url  search_page  ...  \
0                                                NaN          1.0  ...   
1               

In [2]:
x = df_encoded.drop(
    columns=["price_eur", "asin", "title", "product_url", "search_page"],
    errors="ignore"
)

y = df_encoded["price_eur"]

In [63]:
corr = df_encoded.corr(numeric_only=True)["price_eur"].drop("price_eur")

corr = corr.abs().sort_values(ascending=False)

print(corr.head(20))

cpu_Intel Core Ultra 7      0.523376
resolution_2048x1280        0.523376
cpu_Intel Core 5 210H       0.467335
cpu_Intel Core i9-13900H    0.357838
storage_gb                  0.357217
gpu_AMD Radeon 840M         0.256311
ram_gb                      0.150624
gaming                      0.139751
gpu_Intel Iris Xe           0.136314
brand_ASUS                  0.132760
cpu_AMD Ryzen AI 5          0.128123
brand_Acer                  0.119478
gpu_Intel UHD Graphics      0.101533
cpu_Intel Core Ultra 5      0.100787
gpu_Intel Arc Grafik        0.097433
resolution_1920x1200        0.093639
resolution_1920x1080        0.089135
keyboard_QWERTZ             0.088172
cpu_AMD Ryzen AI 7          0.087644
brand_HP                    0.082084
Name: price_eur, dtype: float64


In [54]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(
    score_func=f_regression,
    k=20
)

selector.fit(x, y)

scores = pd.DataFrame({
    "feature": x.columns,
    "score": selector.scores_
})

scores = scores.sort_values("score", ascending=False)

print(scores.head(20))

                      feature       score
282      resolution_2048x1280  104.124824
131    cpu_Intel Core Ultra 7  104.124824
115     cpu_Intel Core 5 210H   77.122673
188  cpu_Intel Core i9-13900H   40.531243
3                  storage_gb   40.370042
207       gpu_AMD Radeon 840M   19.406798
2                      ram_gb    6.407132
6                      gaming    5.497777
226         gpu_Intel Iris Xe    5.225580
8                  brand_ASUS    4.951817
91         cpu_AMD Ryzen AI 5    4.606323
9                  brand_Acer    3.996929
227    gpu_Intel UHD Graphics    2.874920
124    cpu_Intel Core Ultra 5    2.832371
221      gpu_Intel Arc Grafik    2.645213
280      resolution_1920x1200    2.441426
279      resolution_1920x1080    2.210386
275           keyboard_QWERTZ    2.162526
92         cpu_AMD Ryzen AI 7    2.136510
26                   brand_HP    1.872239


In [55]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rfe = RFE(
    estimator=model,
    n_features_to_select=20
)

rfe.fit(x, y)

rfe_result = pd.DataFrame({
    "feature": x.columns,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
})

print(rfe_result.sort_values("ranking").head(20))

                      feature  selected  ranking
0                      rating      True        1
1                     reviews      True        1
3                  storage_gb      True        1
4            screen_size_inch      True        1
6                      gaming      True        1
26                   brand_HP      True        1
31               brand_Lenovo      True        1
267        os_Windows 11 Home      True        1
266             os_Windows 11      True        1
254               gpu_Unknown      True        1
124    cpu_Intel Core Ultra 5      True        1
115     cpu_Intel Core 5 210H      True        1
92         cpu_AMD Ryzen AI 7      True        1
91         cpu_AMD Ryzen AI 5      True        1
207       gpu_AMD Radeon 840M      True        1
226         gpu_Intel Iris Xe      True        1
188  cpu_Intel Core i9-13900H      True        1
131    cpu_Intel Core Ultra 7      True        1
282      resolution_2048x1280      True        1
280      resolution_

In [56]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(x, y)

importance = pd.DataFrame({
    "feature": x.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance.head(20))

                      feature  importance
3                  storage_gb    0.187954
115     cpu_Intel Core 5 210H    0.132690
131    cpu_Intel Core Ultra 7    0.117356
282      resolution_2048x1280    0.113323
188  cpu_Intel Core i9-13900H    0.085085
207       gpu_AMD Radeon 840M    0.080396
0                      rating    0.071982
266             os_Windows 11    0.028253
280      resolution_1920x1200    0.028092
124    cpu_Intel Core Ultra 5    0.017222
6                      gaming    0.015418
31               brand_Lenovo    0.014787
26                   brand_HP    0.012897
267        os_Windows 11 Home    0.012589
1                     reviews    0.012545
91         cpu_AMD Ryzen AI 5    0.010951
4            screen_size_inch    0.010199
254               gpu_Unknown    0.008018
226         gpu_Intel Iris Xe    0.006247
92         cpu_AMD Ryzen AI 7    0.003954


In [57]:
# اهمیت ویژگی‌های Random Forest
importance = pd.DataFrame({
    "feature": x.columns,
    "importance": model.feature_importances_
})

# تعیین گروه هر ویژگی
def get_group(feature):

    f = feature.lower()

    # CPU
    cpu_keywords = [
        "cpu_", "intel", "core i", "core ultra",
        "core 3", "core 5", "core 7", "core 9",
        "ryzen", "celeron", "pentium",
        "snapdragon", "athlon"
    ]

    if any(x in f for x in cpu_keywords):
        return "CPU"

    # GPU
    gpu_keywords = [
        "gpu_", "nvidia", "geforce", "rtx", "gtx",
        "radeon", "iris", "uhd graphics",
        "intel graphics", "arc grafik", "mali", "adreno"
    ]

    if any(x in f for x in gpu_keywords):
        return "GPU"

    # Brand
    brand_keywords = [
        "brand_", "asus", "acer", "hp", "lenovo",
        "dell", "msi", "apple", "microsoft",
        "samsung", "huawei", "lg", "panasonic",
        "medion", "acer"
    ]

    if any(x in f for x in brand_keywords):
        return "Brand"

    # Resolution
    resolution_keywords = [
        "resolution_", "1920x1080", "1920x1200",
        "2048x1280", "2560x1600", "2560x1440",
        "2880x1800", "3840x2160",
        "full hd", "fhd", "qhd", "uhd"
    ]

    if any(x in f for x in resolution_keywords):
        return "Resolution"

    # Numeric features
    if feature == "ram_gb":
        return "RAM"

    if feature == "storage_gb":
        return "Storage"

    if feature == "screen_size_inch":
        return "Screen Size"

    if feature == "rating":
        return "Rating"

    if feature == "reviews":
        return "Reviews"

    if feature == "gaming":
        return "Gaming"

    return "Other"
importance["group"] = importance["feature"].apply(get_group)

# جمع اهمیت هر گروه
group_importance = (
    importance
    .groupby("group")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print(group_importance)

group
CPU            0.381796
Storage        0.187954
Resolution     0.144792
GPU            0.096756
Rating         0.071982
Other          0.045470
Brand          0.031393
Gaming         0.015418
Reviews        0.012545
Screen Size    0.010199
RAM            0.001694
Name: importance, dtype: float64


In [61]:
corr = df_encoded.corr(numeric_only=True)["price_eur"].drop("price_eur")
corr = corr.abs()

corr_df = corr.reset_index()
corr_df.columns = ["feature", "score"]

corr_df["group"] = corr_df["feature"].apply(get_group)

corr_group = (
    corr_df
    .groupby("group")["score"]
    .sum()
    .sort_values(ascending=False)
)

print(corr_group)

group
CPU            3.866734
Brand          1.119898
GPU            0.999717
Resolution     0.921477
Other          0.666457
Storage        0.357217
RAM            0.150624
Gaming         0.139751
Reviews        0.058091
Screen Size    0.019647
Rating         0.009208
Name: score, dtype: float64


In [59]:
kbest_df = pd.DataFrame({
    "feature": x.columns,
    "score": selector.scores_
})

kbest_df["group"] = kbest_df["feature"].apply(get_group)

kbest_group = (
    kbest_df
    .groupby("group")["score"]
    .sum()
    .sort_values(ascending=False)
)

print(kbest_group)

group
CPU            249.998804
Resolution     109.864422
Storage         40.370042
GPU             24.874674
Brand           16.307604
Other            8.172579
RAM              6.407132
Gaming           5.497777
Reviews          0.934548
Screen Size      0.106578
Rating           0.023404
Name: score, dtype: float64


In [60]:
rfe_df = pd.DataFrame({
    "feature": x.columns,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
})

rfe_df["group"] = rfe_df["feature"].apply(get_group)

rfe_group = (
    rfe_df[rfe_df["selected"] == True]
    .groupby("group")
    .size()
    .sort_values(ascending=False)
)

print(rfe_group)

group
CPU            7
Brand          2
GPU            2
Other          2
Resolution     2
Gaming         1
Rating         1
Reviews        1
Screen Size    1
Storage        1
dtype: int64


In [69]:
from sklearn.feature_selection import mutual_info_regression

mi = mutual_info_regression(
    x,
    y,
    random_state=42
)

mi_scores = pd.DataFrame({
    "feature": x.columns,
    "score": mi
}).sort_values(
    "score",
    ascending=False
)

print(mi_scores.head(20))

                         feature     score
3                     storage_gb  0.402253
2                         ram_gb  0.333938
4               screen_size_inch  0.297711
224         resolution_1920x1080  0.130457
234         resolution_2560x1600  0.129557
185       gpu_Intel UHD Graphics  0.126358
225         resolution_1920x1200  0.092644
194  gpu_NVIDIA GeForce RTX 5070  0.091423
212           os_Windows 11 Home  0.089285
193  gpu_NVIDIA GeForce RTX 5060  0.079363
6                         gaming  0.071761
1                        reviews  0.064470
11          brand_Amazon Renewed  0.057215
223          resolution_1366x768  0.056822
9                     brand_Acer  0.054598
182       gpu_Intel Arc Graphics  0.046749
102        cpu_Intel Core 7 240H  0.043462
221             keyboard_Unknown  0.041195
0                         rating  0.036340
163          gpu_AMD Radeon 610M  0.035261
